# Querying a Trained GMS Expert Store

This notebook shows how to load a trained `GMSExpertStore` from disk and exercise its geometric primitives. The same material is summarized in §3.6 of `knowlytix/harness/testing/USER_GUIDE.md`; here we run every primitive against a real store so the contracts are concrete.

By the end you will have:

1. Loaded the monograph's Chapter 4 ground-truth store from `gmsh_testing_ext/lab_stores/ch04_ground_truth/`.
2. Exercised the four primitives that drive verification: `score_triple`, `query_triples`, `lookup_enm`, `link_predict`.
3. Sampled the two structural primitives: `tension_energy` (entity disagreement) and `check_holonomy` (multi-hop consistency).
4. Built your own store from a tiny inline markdown document via `DOEGMSBenchmark.ingest()` and queried it with the same API.

All cells run on CPU.

## 1. Preamble

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import torch

# Make sure the kg-memory project root is importable.
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name and not (PROJECT_ROOT / "harness" / "doe" / "__init__.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

device = torch.device("cpu")  # querying is fast on CPU; load() is tiny
print(f"project root: {PROJECT_ROOT}")
print(f"device:       {device}")

## 2. Load an existing store from disk

A trained store is a directory of eight files. Anyone who has run the Chapter 4 lab (or `gmsh_testing_ext/scripts/build_ch04_fixture.py`) already has one ready at `gmsh_testing_ext/lab_stores/ch04_ground_truth/`. Inspect the layout first.

In [2]:
store_path = PROJECT_ROOT / "gmsh_testing_ext" / "lab_stores" / "ch04_ground_truth"

print(f"{store_path}/")
for entry in sorted(store_path.iterdir()):
    if entry.is_file():
        size_kb = entry.stat().st_size / 1024
        print(f"  {entry.name:20s}  {size_kb:8.1f} KB")
    else:
        print(f"  {entry.name}/")

/home/asudjianto/jupyterlab/kg-memory/gmsh_testing_ext/lab_stores/ch04_ground_truth/
  adapter.json              28.4 KB
  documents/
  enm.json                  27.6 KB
  metadata.json              0.1 KB
  model.pt                5540.4 KB
  model_dims.json            0.1 KB
  phase.json                 0.4 KB
  triples.json              37.9 KB


| File | Contents |
|------|----------|
| `model.pt` | PyTorch state dict — entity embeddings, Cayley rotors, projection matrices |
| `model_dims.json` | Number of entities and relations (needed to reconstruct the model) |
| `adapter.json` | Entity- and relation-name vocabularies |
| `enm.json` | Exact numeric values (SHA-256 verified) |
| `triples.json` | All knowledge triples extracted from the source document |
| `phase.json` | Phase encoder ranges per metric type |
| `metadata.json` | Ingestion history, timestamps |
| `documents/combined.md` | Original source document (provenance) |

Loading is two lines: a `DocGMSConfig` pointing at the directory, and `store.load()` which reads everything back into memory.

In [ ]:
from knowlytix.knowledge.config import DocGMSConfig
from knowlytix.knowledge.store import GMSExpertStore

config = DocGMSConfig(store_path=str(store_path))
store = GMSExpertStore(config, device=device)
ok = store.load()
assert ok, f"no trained store at {store_path}"

stats = store.stats()
print("\nStore stats:")
for k, v in stats.items():
    print(f"  {k:12s} {v}")

## 3. The four primitives that drive verification

Every typed verdict in the harness reduces to one of six geometric primitives (Core Concepts §3 in `USER_GUIDE.md`). Four of them are demonstrated below on the Chapter 4 model risk corpus; the last two are sampled in §4.

### 3.1 `score_triple(h, r, t)` — fact verification

Returns the geodesic distance between the rotor-transported head and the tail embedding on $S^{m-1}$. **Lower means more plausible**; a value near 0 means the model believes `(h, r, t)` is a stored fact. The Chapter 4 store contains `(MDL-001, has_test_auc, 0.8412)` as ground truth — let's see what the geometry says about it, and contrast with another model's AUC plugged in as a wrong tail.

In [4]:
head = "MDL-001"
rel = "has_test_auc"

candidates = [
    ("0.8412", "truth — MDL-001 test AUC"),
    ("0.8934", "MDL-001 train AUC (right model, wrong relation tail)"),
    ("0.9234", "MDL-002 test AUC (wrong model)"),
    ("0.7234", "MDL-005 test AUC (wrong model)"),
]

print(f"score_triple({head!r}, {rel!r}, ?)  — lower = more plausible\n")
for tail, note in candidates:
    score = store.score_triple(head, rel, tail)
    print(f"  tail={tail:8s}  score={score:7.4f}    {note}")

score_triple('MDL-001', 'has_test_auc', ?)  — lower = more plausible

  tail=0.8412    score= 0.0770    truth — MDL-001 test AUC
  tail=0.8934    score= 1.3810    MDL-001 train AUC (right model, wrong relation tail)
  tail=0.9234    score= 1.5539    MDL-002 test AUC (wrong model)
  tail=0.7234    score= 1.3758    MDL-005 test AUC (wrong model)


The truth scores roughly an order of magnitude below the distractors — that gap is what the harness's `GraphVerifier` and `HallucinationOracle` rely on. Note that `score_triple` returns `None` if the head, relation, or tail is not in the vocabulary; verifiers treat that as "unscoreable", not "wrong".

### 3.2 `query_triples(head, relation, tail)` — graph pattern lookup

Pure structural query against the triple store, no geometry involved. Any of the three slots can be omitted (`None`) to act as a wildcard. This is what tool-ordering and policy verifiers use when they need to ask *what is recorded*, not *what is plausible*.

In [5]:
# Pattern 1: everything we know about MDL-001 with relation has_status
print("query_triples(head='MDL-001', relation='has_status'):")
for h, r, t in store.query_triples(head="MDL-001", relation="has_status"):
    print(f"  ({h}, {r}, {t})")

# Pattern 2: every model's status (relation as the only filter)
print("\nquery_triples(relation='has_status'):  (first 6)")
for h, r, t in list(store.query_triples(relation="has_status"))[:6]:
    print(f"  ({h}, {r}, {t})")

# Pattern 3: every triple about MDL-001 (head as the only filter)
mdl001 = list(store.query_triples(head="MDL-001"))
print(f"\nquery_triples(head='MDL-001'):  {len(mdl001)} triples")
for h, r, t in mdl001[:5]:
    print(f"  ({h}, {r}, {t})")

query_triples(head='MDL-001', relation='has_status'):
  (MDL-001, has_status, Degraded)

query_triples(relation='has_status'):  (first 6)
  (MDL-001, has_status, Degraded)
  (MDL-002, has_status, Watch)
  (MDL-003, has_status, Degraded)
  (MDL-004, has_status, Stable)
  (MDL-005, has_status, Degraded)
  (MDL-006, has_status, Watch)

query_triples(head='MDL-001'):  82 triples
  (MDL-001, has_tier, 1.0)
  (MDL-001, has_model_name, CreditScore-XGB)
  (MDL-001, has_domain, Credit Decisioning)
  (MDL-001, has_risk_rating, High)
  (MDL-001, has_status, Degraded)


### 3.3 `lookup_enm(category, entity_id)` — exact numeric retrieval

Geometric scoring is good for plausibility but it is *approximate* — embeddings live on a unit sphere, so AUC=0.8412 and AUC=0.8413 may sit on top of each other. When verification needs the actual digit-for-digit value (regulatory reporting, capital impact, audit replay), the harness reads from the **Exact Numeric Memory** instead. Each entry is keyed by `(category, entity_id)` and stored with a SHA-256 integrity check.

In [6]:
lookups = [
    ("performance_metrics", "MDL-001/Train AUC"),
    ("performance_metrics", "MDL-001/Test AUC"),
    ("performance_metrics", "MDL-001/Gini"),
    ("performance_metrics", "MDL-001/KS Statistic"),
]
for category, eid in lookups:
    val = store.lookup_enm(category, eid)
    print(f"  {category}::{eid:30s}  =  {val}")

  performance_metrics::MDL-001/Train AUC               =  0.8934
  performance_metrics::MDL-001/Test AUC                =  0.8412
  performance_metrics::MDL-001/Gini                    =  0.6824
  performance_metrics::MDL-001/KS Statistic            =  0.4891


### 3.4 `link_predict(head, relation, top_k)` — top-k completion

Given a head and relation, score every entity in the vocabulary as a candidate tail and return the top-k by geodesic distance. Used by the harness when generating distractor answers and when ranking competing claims. `link_predict` applies a type constraint — only entities that have appeared as a tail for this relation in the training graph are considered, so numeric values cannot pollute categorical predictions and vice versa.

In [7]:
print("link_predict('MDL-001', 'has_test_auc', top_k=5):\n")
print(f"  {'rank':>4s}  {'tail':>10s}  {'score':>7s}")
for i, (tail, score) in enumerate(store.link_predict("MDL-001", "has_test_auc", top_k=5), 1):
    marker = "  <- ground truth" if tail == "0.8412" else ""
    print(f"  {i:>4d}  {tail:>10s}  {score:>7.4f}{marker}")

link_predict('MDL-001', 'has_test_auc', top_k=5):

  rank        tail    score


     1      0.8412   0.3023  <- ground truth
     2      0.8345   1.2955
     3      0.7234   1.3202
     4      0.8756   1.4393
     5      0.8567   1.4633


## 4. Structural primitives: tension and holonomy

Two more primitives are exposed for cross-claim and multi-hop checks. They are demonstrated briefly here; their main use sites are `ContradictionGenerator`, `RecoveryEvaluator`, and the multi-hop verifier.

In [8]:
# 4.1 tension_energy(a, b): entity-level disagreement.
# 0 = aligned, ~sqrt(2) = unrelated, 2 = directly opposed.
print("tension_energy:")
for a, b in [("MDL-001", "MDL-004"), ("MDL-001", "MDL-003"), ("MDL-001", "MDL-001")]:
    te = store.tension_energy(a, b)
    print(f"  {a} vs {b}:  {te:.4f}")

# 4.2 check_holonomy(path, direct): multi-hop transport defect.
# 0 = composed path agrees with the direct edge.
# Defects > 0 are the signal a process-evaluator turns into a PROCESS_DEVIATION.
print("\ncheck_holonomy:")
defect = store.check_holonomy(["has_train_auc"], "has_test_auc")
print(f"  [has_train_auc] vs has_test_auc:  {defect:.4f}")

tension_energy:
  MDL-001 vs MDL-004:  0.0143
  MDL-001 vs MDL-003:  0.0227
  MDL-001 vs MDL-001:  0.0001

check_holonomy:
  [has_train_auc] vs has_test_auc:  10.9798


## 5. Build your own store

When you do not have a pre-trained store sitting around, the harness's `DOEGMSBenchmark.ingest()` wraps the full ingestion pipeline (graph extraction → embedding → ENM population → save). Below we build a tiny store from an inline markdown document, save it, then query it with the same primitives.

On a small document like this, ingestion takes a few seconds on CPU.

In [ ]:
import tempfile

from knowlytix.harness.testing import DOEGMSBenchmark, DOEHarnessConfig

TINY_DOC = """# Quarterly Model Review

## Models

| model_id | algorithm | test_auc | status |
|----------|-----------|----------|--------|
| MDL-A    | XGBoost   | 0.8412   | Stable |
| MDL-B    | LightGBM  | 0.7956   | Watch  |
| MDL-C    | RandomForest | 0.8201 | Stable |
"""

tmp = Path(tempfile.mkdtemp(prefix="qgms_"))
md_path = tmp / "tiny_review.md"
md_path.write_text(TINY_DOC)

tiny_cfg = DOEHarnessConfig(
    markdown_path=str(md_path),
    stores_dir=str(tmp / "stores"),
    n_runs=1,           # we are not running a benchmark, just ingesting
    factor_group="quick_screen",
)
bench = DOEGMSBenchmark(tiny_cfg)
tiny_store = bench.ingest()
print("\nTiny store stats:", tiny_store.stats())

In [10]:
# Same primitives, fresh store.
print("query_triples(head='MDL-A'):")
for h, r, t in tiny_store.query_triples(head="MDL-A"):
    print(f"  ({h}, {r}, {t})")

score_truth = tiny_store.score_triple("MDL-A", "has_test_auc", "0.8412")
score_wrong = tiny_store.score_triple("MDL-A", "has_test_auc", "0.7956")
print(f"\nscore_triple MDL-A has_test_auc 0.8412 (truth):  {score_truth}")
print(f"score_triple MDL-A has_test_auc 0.7956 (wrong):  {score_wrong}")

query_triples(head='MDL-A'):
  (MDL-A, has_test_auc, 0.8412)
  (MDL-A, has_algorithm, XGBoost)
  (MDL-A, has_status, Stable)
  (MDL-A, in_section, models)

score_triple MDL-A has_test_auc 0.8412 (truth):  0.3720638155937195
score_triple MDL-A has_test_auc 0.7956 (wrong):  1.499452829360962


## 6. Where to go next

- **Use the primitives inside an evaluator** → §4 (Tutorial 1) of `USER_GUIDE.md`.
- **Wrap them as a judge** → `gmsh_testing_ext/notebooks/ch06_geometric_judge.ipynb`.
- **Generate questions whose answers reduce to these primitives** → §17 (Taxonomy Generators) and the geometric generator family.
- **Train a fresh store from a document of your own** → `gmsh_testing_ext/notebooks/ch04_building_ground_truth.ipynb` walks through the full assembly pipeline, not just the `bench.ingest()` shortcut used in §5 above.

Every typed verdict the harness emits is some combination of the calls demonstrated above; once they read naturally, the rest of the platform is composition.